# Phase 6 — Business Digital Twin Validation Notebook

This notebook validates the **production** `backend/simulation` package using local
synthetic data only.

**No network access, no API keys, and no production logic in this notebook** —
every computation is a call into `backend/simulation/*`, `backend/risk_engine/*`
or `backend/data_engine/*`.

Contents:

1. Setup / path bootstrap
2. The frozen §1 parameter table, with which values were real and which assumed
3. Starting state (t = 0) and the month grid
4. **Month 1, calculated by hand** and compared to the engine
5. Month 2 progression
6. Cash reconciliation across the horizon
7. The frozen §6 invariants
8. Deterministic repeatability
9. Edge case: a funding gap and negative cash
10. Edge case: a shrinking business releasing working capital
11. Output shaped for Phase 8 stress testing
12. What this notebook does **not** claim

A simulation is an *assumption-based intervention on a model*, not discovered
causal truth, and never professional financial advice.

In [1]:
import sys
from pathlib import Path

# Path bootstrap so a fresh kernel can import the repository package.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "backend").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from backend.data_engine.contracts import SyntheticCompanyConfig
from backend.data_engine.ingest.synthetic import generate_company
from backend.simulation.assumptions import derive_assumptions
from backend.simulation.contracts import (
    MAX_HORIZON_MONTHS,
    TWIN_VERSION,
    SimulationInput,
    SimulationOverrides,
)
from backend.simulation.twin import (
    build_initial_state_from_period,
    run_twin,
    simulate,
)

print("twin version:", TWIN_VERSION, "| frozen horizon max:", MAX_HORIZON_MONTHS)

twin version: 1.0.0 | frozen horizon max: 36


## 1. A deterministic canonical company

The synthetic generator is the project's own offline data path (R4). Seed 1001
yields 24 **monthly** periods, so no annual-to-monthly conversion is involved
anywhere in this notebook.

In [2]:
dataset = generate_company(SyntheticCompanyConfig(seed=1001, periods=24))
latest = dataset.periods[-1]

print("company :", dataset.profile.name, "|", dataset.profile.sector.value)
print("currency:", dataset.profile.currency)
print("periods :", len(dataset.periods), "x", dataset.periods[0].frequency.value)
print("window  :", dataset.periods[0].period_end, "->", latest.period_end)

company : Greystone Industries | manufacturing
currency: INR
periods : 24 x monthly
window  : 2023-01-31 -> 2024-12-31


## 2. The frozen §1 parameter table

`derive_assumptions` reports every parameter with its provenance tag and whether
a documented fallback had to be used. A `real` value was measured from the
statements; an `assumption` value is a stated default. Nothing is invented
silently.

In [3]:
assumptions, records, fallbacks = derive_assumptions(dataset)

print(f"{'parameter':<24}{'value':>16}  {'tag':<11} fallback")
print("-" * 66)
for record in records:
    shown = f"{record.value:,.4f}" if abs(record.value) < 1000 else f"{record.value:,.0f}"
    print(f"{record.parameter:<24}{shown:>16}  {record.tag.value:<11} {record.fallback_used}")

print()
print("fallbacks used:", fallbacks)

parameter                          value  tag         fallback
------------------------------------------------------------------
growth_rate                       0.0440  real        False
cogs_ratio                        0.7012  real        False
fixed_cost                       642,263  assumption  True
variable_cost_ratio               0.1057  assumption  True
da_monthly                       284,998  real        False
tax_rate                          0.2500  real        False
base_rate                         0.0600  assumption  True
floating_debt_share               0.5892  real        False
fx_import_cost_share              0.1519  real        False
fx_revenue_share                  0.1313  real        False
commodity_cost_share              0.2836  real        False
dso                              46.1430  real        False
dio                              59.0218  real        False
dpo                              49.4515  real        False
capex_monthly                    

Two groups of fallback fire, both mandated by the frozen table in
`docs/01_architecture/simulation.md` §1:

* **`base_rate`** — `r₀ = FEDFUNDS latest + 200 bp spread, offline fallback 6%`.
  The twin never fetches, so the offline default stands. No network access is
  needed or used.
* **`fixed_cost` / `variable_cost_ratio`** — the OLS fit of opex on revenue fell
  back to `F = 0.5·opex₀`, `v = 0.5·opex₀/rev₀`. That happens when the fitted
  slope would be negative, which the frozen `F, v ≥ 0` constraint forbids.

Everything else was measured from the 12-month trailing window.

## 3. Starting state and the month grid

t = 0 is the last observed period. Simulation months advance on month-end dates
(month-end convention, decision D-2) — the same grid `PeriodFinancials` uses, so
leap years need no special handling.

In [4]:
initial = build_initial_state_from_period(latest, dataset)
result = simulate(dataset, horizon_months=12)

print("t = 0 (last observed):", initial.period_end)
print("opening revenue      : {:,.2f}".format(initial.revenue))
print("opening cash         : {:,.2f}".format(initial.cash))
print("opening debt         : {:,.2f}".format(initial.debt))
print("opening NWC          : {:,.2f}".format(initial.nwc))
print()
print("simulated months     :", result.horizon_months)
print("first simulated month:", result.months[0].period_end)
print("last simulated month :", result.end_period_end)
print("month ends unique    :", len({m.period_end for m in result.months}) == 12)

t = 0 (last observed): 2024-12-31
opening revenue      : 6,318,610.76
opening cash         : 14,218,735.40
opening debt         : 17,209,502.71
opening NWC          : 11,132,120.82

simulated months     : 12
first simulated month: 2025-01-31
last simulated month : 2025-12-31
month ends unique    : True


## 4. Month 1, calculated by hand

Every frozen §2 line is worked out by hand below and then compared with the
engine. This is the strongest evidence that the recursion is what the frozen
specification says it is.

In [5]:
A = assumptions
m1 = result.months[0]

# --- demand and supply channels (frozen §2); no scenario deltas are applied ---
monthly_growth = (1.0 + A.growth_rate) ** (1.0 / 12.0)
rev_dem = initial.revenue * monthly_growth
rev_dem -= 0.0 * A.fx_revenue_share * A.fx_demand_elasticity * rev_dem
rev_sup = A.revenue_capacity
rev_hand = min(rev_dem, rev_sup)

# --- cost of goods: commodity and FX legs, both damped by (1 - ptc) ---
ptc_effect = 1.0 - A.pass_through
cogs_hand = rev_hand * A.cogs_ratio
cogs_hand *= 1.0 + 0.0 * A.commodity_cost_share * ptc_effect
cogs_hand *= 1.0 + 0.0 * A.fx_import_cost_share * ptc_effect
cogs_hand *= 1.0 + 0.0

# --- operating result ---
gp_hand = rev_hand - cogs_hand
opex_hand = (A.fixed_cost + A.variable_cost_ratio * rev_hand) * 1.0
ebitda_hand = gp_hand - opex_hand
ebit_hand = ebitda_hand - A.da_monthly

# --- interest on the OPENING debt, split floating / fixed ---
opening_debt = A.initial_debt
interest_hand = (A.base_rate / 12.0) * opening_debt * A.floating_debt_share
interest_hand += (A.base_rate / 12.0) * opening_debt * (1.0 - A.floating_debt_share)

ebt_hand = ebit_hand - interest_hand
tax_hand = max(0.0, ebt_hand) * A.tax_rate
ni_hand = ebt_hand - tax_hand

# --- working capital and operating cash flow ---
ar_hand = rev_hand * A.dso / 30.0
inv_hand = cogs_hand * A.dio / 30.0
ap_hand = cogs_hand * A.dpo / 30.0
nwc_hand = ar_hand + inv_hand - ap_hand
ocf_hand = ni_hand + A.da_monthly - (nwc_hand - initial.nwc)

# --- investing and financing ---
capex_hand = A.capex_monthly
principal_hand = A.principal_for_month(1)
cash_before_draws = initial.cash + ocf_hand - capex_hand - principal_hand
shortfall = max(0.0, A.min_cash_buffer - cash_before_draws)
draws_hand = min(shortfall, A.revolver_cap)
cash_hand = cash_before_draws + draws_hand
debt_hand = max(0.0, opening_debt - principal_hand + draws_hand)

rows = [
    ("revenue", rev_hand, m1.revenue),
    ("cogs", cogs_hand, m1.cogs),
    ("gross profit", gp_hand, m1.gross_profit),
    ("opex", opex_hand, m1.opex),
    ("ebitda", ebitda_hand, m1.ebitda),
    ("interest", interest_hand, m1.interest_expense),
    ("tax", tax_hand, m1.tax),
    ("net income", ni_hand, m1.net_income),
    ("operating CF", ocf_hand, m1.ocf),
    ("draws", draws_hand, m1.draws),
    ("CASH", cash_hand, m1.cash),
    ("DEBT", debt_hand, m1.total_debt),
]
print(f"{'line':<14}{'by hand':>18}{'engine':>18}")
print("-" * 50)
for name, hand, engine in rows:
    print(f"{name:<14}{hand:>18,.2f}{engine:>18,.2f}")

line                     by hand            engine
--------------------------------------------------
revenue             6,336,413.55      6,336,413.55
cogs                4,443,289.88      4,443,289.88
gross profit        1,893,123.67      1,893,123.67
opex                1,312,254.25      1,312,254.25
ebitda                580,869.42        580,869.42
interest               86,047.51         86,047.51
tax                    52,455.86         52,455.86
net income            157,367.59        157,367.59
operating CF          411,001.10        411,001.10
draws                       0.00              0.00
CASH               13,684,726.29     13,684,726.29
DEBT               16,461,914.38     16,461,914.38


In [6]:
# Assert the hand calculation rather than eyeballing it.
checks = {
    "revenue": (rev_hand, m1.revenue),
    "cogs": (cogs_hand, m1.cogs),
    "gross_profit": (gp_hand, m1.gross_profit),
    "opex": (opex_hand, m1.opex),
    "ebitda": (ebitda_hand, m1.ebitda),
    "ebit": (ebit_hand, m1.ebit),
    "interest": (interest_hand, m1.interest_expense),
    "ebt": (ebt_hand, m1.ebt),
    "tax": (tax_hand, m1.tax),
    "net_income": (ni_hand, m1.net_income),
    "nwc": (nwc_hand, m1.nwc),
    "ocf": (ocf_hand, m1.ocf),
    "cash": (cash_hand, m1.cash),
    "total_debt": (debt_hand, m1.total_debt),
}
worst = 0.0
for name, (hand, engine) in checks.items():
    scale = max(abs(hand), abs(engine), 1.0)
    worst = max(worst, abs(hand - engine) / scale)

print(f"{len(checks)} month-1 lines agree with the engine")
print(f"worst relative residual: {worst:.3e}  (tolerance 1e-9)")
assert worst < 1e-9

14 month-1 lines agree with the engine
worst relative residual: 1.691e-16  (tolerance 1e-9)


## 5. Month 2 progression

Month 2 opens on month 1's closing balances. Note that the frozen recursion
applies each delta to the **previous** month's revenue, so a persistent shock
compounds month over month rather than resetting to a level.

In [7]:
print(f"{'month':<9}{'revenue':>15}{'ebitda':>13}{'cash':>15}{'debt':>15}")
for m in result.months[:4]:
    print(
        f"{m.period_end!s:<9}{m.revenue:>15,.0f}{m.ebitda:>13,.0f}"
        f"{m.cash:>15,.0f}{m.total_debt:>15,.0f}"
    )

first, second = result.months[0], result.months[1]
print()
print("month 2 starts from month 1 revenue:", f"{first.revenue:,.2f}", "->", f"{second.revenue:,.2f}")
print("month 2 opening cash == month 1 closing cash:", second.cash > first.cash)

month            revenue       ebitda           cash           debt
2025-01-31      6,336,414      580,869     13,684,726     16,461,914
2025-02-28      6,336,414      580,869     13,184,886     15,714,326
2025-03-31      6,336,414      580,869     12,687,848     14,966,738
2025-04-30      6,336,414      580,869     12,692,007     14,717,542

month 2 starts from month 1 revenue: 6,336,413.55 -> 6,336,413.55
month 2 opening cash == month 1 closing cash: False


## 6. Cash reconciliation across the horizon

The frozen §6.1 identity is
`Cash_t = Cash_{t-1} + OCF_t − Capex_t − principal_t + draws_t − one_off_cost`
(±1e-6 relative). It is recomputed here from the emitted months, not read back
from the engine's internal variables.

In [8]:
previous = initial.cash
print(f"{'m':<3}{'OCF':>15}{'capex':>13}{'principal':>12}{'draws':>12}{'one-off':>11}{'cash':>16}")
for m in result.months:
    print(
        f"{m.period_index:<3}{m.ocf:>15,.0f}{-m.capex:>13,.0f}{-m.principal:>12,.0f}"
        f"{m.draws:>12,.0f}{-m.one_off_cost:>11,.0f}{m.cash:>16,.0f}"
    )
    expected = previous + m.ocf - m.capex - m.principal + m.draws - m.one_off_cost
    assert abs(m.cash - expected) <= 1e-6 * max(abs(m.cash), 1.0)
    previous = m.cash

print()
print("cash identity holds for every month (tolerance 1e-6 relative)")

m              OCF        capex   principal       draws    one-off            cash
1          411,001     -197,422    -747,588           0         -0      13,684,726
2          445,170     -197,422    -747,588           0         -0      13,184,886
3          447,973     -197,422    -747,588           0         -0      12,687,848
4          450,776     -197,422    -249,196           0         -0      12,692,007
5          451,711     -197,422    -249,196           0         -0      12,697,100
6          452,645     -197,422    -249,196           0         -0      12,703,127
7          453,580     -197,422    -249,196           0         -0      12,710,089
8          454,514     -197,422    -249,196           0         -0      12,717,985
9          455,449     -197,422    -249,196           0         -0      12,726,816
10         456,383     -197,422    -249,196           0         -0      12,736,581
11         457,318     -197,422    -249,196           0         -0      12,747,281
12  

## 7. The frozen §6 invariants

Six identities are checked per run inside `check_invariants`, computed from the
emitted months so they are genuine checks rather than restatements of the same
arithmetic.

In [9]:
print(f"{'invariant':<36}{'holds':>8}{'residual':>14}{'tolerance':>14}")
print("-" * 72)
for inv in result.invariants:
    print(f"{inv.name:<36}{str(inv.holds):>8}{inv.residual:>14.3e}{inv.tolerance:>14.3e}")

print()
print("all invariants hold:", result.invariants_hold)

invariant                              holds      residual     tolerance
------------------------------------------------------------------------
cash_identity                           True     0.000e+00     1.000e-06
debt_roll_forward                       True     0.000e+00     1.000e-09
nwc_continuity                          True     0.000e+00     1.000e-09
tax_non_negative                        True     0.000e+00     0.000e+00
negative_cash_only_via_funding_gap      True     0.000e+00     0.000e+00
horizon_bounded                         True     1.200e+01     3.600e+01

all invariants hold: True


## 8. Deterministic repeatability

Same inputs ⇒ identical outputs (R6, NFR1, frozen §6.2). The twin contains no
random process, so no seed exists and none is invented.

In [10]:
again = simulate(dataset, horizon_months=12)

print("identical model dump:", again.model_dump() == result.model_dump())
print("baseline flag       :", result.provenance.is_baseline)
print("twin version        :", result.provenance.twin_version)
print("risk registry       :", result.provenance.registry_version)
print("generator seeds     :", result.provenance.generator_seeds)
print("history periods used:", result.diagnostics.history_periods)

identical model dump: True
baseline flag       : True
twin version        : 1.0.0
risk registry       : 1.0.0
generator seeds     : [1001]
history periods used: 24


## 9. Edge case — a funding gap, and negative cash

Frozen §2: draws restore the minimum cash buffer `B` up to the revolver cap
`RC`; beyond `RC` the period records a **`funding_gap`** hard breach, and only
then may cash go negative. Cash is never clamped.

In [11]:
stressed_assumptions = assumptions.model_copy(
    update={"min_cash_buffer": 6_000_000.0, "revolver_cap": 10_000_000.0}
)
gapped = run_twin(
    SimulationInput(
        horizon_months=6,
        assumptions=stressed_assumptions,
        initial_state=build_initial_state_from_period(latest, dataset),
        overrides=SimulationOverrides(one_off_cost=20_000_000.0),
    )
)

print(f"{'m':<3}{'cash':>16}{'draws':>14}{'gap':>7}  status")
for m in gapped.months:
    print(
        f"{m.period_index:<3}{m.cash:>16,.0f}{m.draws:>14,.0f}"
        f"{str(m.funding_gap):>7}  {m.status.value}"
    )

print()
print("no draw exceeds the revolver cap:",
      all(m.draws <= stressed_assumptions.revolver_cap + 1e-9 for m in gapped.months))
print("negative cash only via a gap   :",
      all(m.cash >= 0.0 or m.funding_gap for m in gapped.months))
print("invariants still hold          :", gapped.invariants_hold)

m              cash         draws    gap  status
1         3,684,726    10,000,000   True  funding_gap
2         6,000,000     2,852,614  False  ok
3         6,000,000       545,235  False  ok
4         6,000,000        46,084  False  ok
5         6,000,000        45,322  False  ok
6         6,000,000        44,557  False  ok

no draw exceeds the revolver cap: True
negative cash only via a gap   : True
invariants still hold          : True


## 10. Edge case — a shrinking business releases working capital

Counter-intuitive but real, and a property of the frozen model rather than a
bug: `OCF_t = NI_t + DA_t − ΔNWC_t`. When revenue collapses, receivables and
inventory are collected faster than profit falls, so `ΔNWC` is strongly
**negative** and the cash flow statement improves even while the income
statement deteriorates.

In [12]:
shrinking = simulate(
    dataset,
    horizon_months=12,
    overrides=SimulationOverrides(revenue_change=-0.30),
)

print(f"{'m':<3}{'revenue':>15}{'net income':>14}{'dNWC':>14}{'OCF':>14}")
for m in shrinking.months[:5]:
    print(
        f"{m.period_index:<3}{m.revenue:>15,.0f}{m.net_income:>14,.0f}"
        f"{m.delta_nwc:>14,.0f}{m.ocf:>14,.0f}"
    )

losing = [m for m in shrinking.months if m.net_income < 0.0]
print()
print("months with negative net income  :", len(losing))
print("all of them release working cap :", all(m.delta_nwc < 0.0 for m in losing))
print("some still show positive OCF    :", any(m.ocf > 0.0 for m in losing))

m          revenue    net income          dNWC           OCF
1        4,438,920      -156,454    -3,311,637     3,440,181
2        3,118,409      -407,617    -2,326,475     2,203,856
3        2,190,730      -582,951    -1,634,384     1,336,431
4        1,539,021      -705,014    -1,148,180       728,164
5        1,081,186      -792,145      -806,614       299,467

months with negative net income  : 12
all of them release working cap : True
some still show positive OCF    : True


## 11. Output shaped for Phase 8

Phase 6 reports what happened; it does not compare runs or declare a breach.
These are the series and trough values the Phase 8 stress engine will
difference, and the conditions it will turn into breach flags.

In [13]:
s = result.summary
print("horizon months        :", result.horizon_months)
print("min cash              : {:,.2f} (month {})".format(s.min_cash, s.min_cash_period))
print("min DSCR              :", None if s.min_dscr is None else round(s.min_dscr, 4))
print("min interest coverage :",
      None if s.min_interest_coverage is None else round(s.min_interest_coverage, 4))
print("final cash            : {:,.2f}".format(s.final_cash))
print("final debt            : {:,.2f}".format(s.final_debt))
print("lowest EBITDA         : {:,.2f}".format(s.lowest_ebitda))
print("total draws           : {:,.2f}".format(s.total_draws))
print("total principal       : {:,.2f}".format(s.total_principal))
print("funding gap months    :", s.funding_gap_months)
print("revenue-capped months :", s.revenue_capped_months)
print("conditions (not breaches): breached_buffer =", s.breached_buffer,
      "| had_funding_gap =", s.had_funding_gap)
print()
print("series available for differencing:")
for field in ("revenue", "ebitda", "ebit", "net_income", "ocf", "cash", "total_debt"):
    series = result.trajectory(field)
    print(f"  {field:<12}{len(series):>3} points  {series[0]:>14,.0f} -> {series[-1]:>14,.0f}")

horizon months        : 12
min cash              : 12,687,848.32 (month 3)
min DSCR              : 0.6968
min interest coverage : 3.4385
final cash            : 12,758,915.53
final debt            : 12,723,972.77
lowest EBITDA         : 580,869.42
total draws           : 0.00
total principal       : 4,485,529.94
funding gap months    : []
revenue-capped months : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
conditions (not breaches): breached_buffer = False | had_funding_gap = False

series available for differencing:
  revenue      12 points       6,336,414 ->      6,336,414
  ebitda       12 points         580,869 ->        580,869
  ebit         12 points         295,871 ->        295,871
  net_income   12 points         157,368 ->        173,254
  ocf          12 points         411,001 ->        458,252
  cash         12 points      13,684,726 ->     12,758,916
  total_debt   12 points      16,461,914 ->     12,723,973


## 12. What this notebook does **not** claim

* A simulation is an **assumption-based intervention on a model**, not evidence
  about why a business behaves as it does. The parameter table in §2 is the
  honest statement of what was assumed and what was measured.
* It is **not a forecast**, not a valuation and **not professional financial
  advice**. `base_rate` alone is a documented 6% offline default, not a market
  observation.
* The monthly recursion needs **monthly history**. The canonical fixtures
  provide it; a company whose statements are annual or quarterly cannot be fed
  to the twin without a monthlyisation step, which Phase 6 deliberately does
  **not** invent (decision D-1).
* Coverage, breach thresholds, baseline-vs-stressed comparison and the EBITDA
  waterfall are **Phase 8**. This notebook shows only the raw material.
* No Monte Carlo, no stochastic process and no LLM call is involved: the twin
  is a pure function of its inputs.